# B2-019-attention-transformers — Practice p17 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** causal-self-attention, sinusoidal-positional-encoding

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

Each row i applies a lower-triangular mask before softmax, so its hidden state is a weighted combination only of value rows j<=i. The class head at row i therefore cannot use a token after i while predicting target i. The prescribed deterministic run reduces cross-entropy from about 1.39674 to 0.26725 and predicts all five pinned targets.

In [ ]:
import math
import numpy as np
import torch
from torch import nn

SEED = 20260808
ATOL = 1e-8
RTOL = 1e-8
np.random.seed(SEED)
torch.manual_seed(SEED)
X = np.eye(4, dtype=np.float64)[[0, 1, 2, 1, 3, 0]]
positions = np.arange(5, dtype=np.float64)[:, None]
rates = np.array([[1.0, 0.1]], dtype=np.float64)
POSITIONAL = np.empty((5, 4), dtype=np.float64)
POSITIONAL[:, 0::2] = np.sin(positions * rates)
POSITIONAL[:, 1::2] = np.cos(positions * rates)
inputs = torch.tensor(X[:-1] + 0.1 * POSITIONAL, dtype=torch.float64).unsqueeze(0)
targets = torch.tensor([[1, 2, 1, 3, 0]], dtype=torch.long)
allowed = torch.tril(torch.ones(5, 5, dtype=torch.bool))
INITIAL_PARAMETERS = {
    "q.weight": np.array([[.20,-.10,.00,.10],[.00,.30,-.20,.10],[.10,.00,.25,-.15],[-.20,.10,.05,.30]], dtype=np.float64),
    "k.weight": np.array([[.15,.00,-.10,.20],[.10,.25,.00,-.10],[-.05,.10,.30,.00],[.20,-.15,.10,.25]], dtype=np.float64),
    "v.weight": np.array([[.30,.00,.10,-.10],[-.10,.20,.00,.25],[.05,-.20,.35,.00],[.10,.15,-.10,.20]], dtype=np.float64),
    "out.weight": np.array([[.25,-.05,.10,.00],[.00,.30,-.10,.05],[-.15,.05,.20,.10],[.10,-.10,.00,.25]], dtype=np.float64),
    "head.weight": np.array([[.20,.00,-.10,.10],[-.05,.25,.10,.00],[.10,-.15,.30,.05],[.00,.10,-.05,.20]], dtype=np.float64),
    "head.bias": np.array([.01,-.02,.03,-.04], dtype=np.float64),
}

class CausalPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.q = nn.Linear(4, 4, bias=False, dtype=torch.float64)
        self.k = nn.Linear(4, 4, bias=False, dtype=torch.float64)
        self.v = nn.Linear(4, 4, bias=False, dtype=torch.float64)
        self.out = nn.Linear(4, 4, bias=False, dtype=torch.float64)
        self.head = nn.Linear(4, 4, bias=True, dtype=torch.float64)
    def forward(self, inputs, allowed):
        q, k, v = self.q(inputs), self.k(inputs), self.v(inputs)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(4)
        weights = torch.softmax(scores.masked_fill(~allowed, float("-inf")), dim=-1)
        return self.head(self.out(torch.matmul(weights, v)))

model = CausalPredictor()
with torch.no_grad():
    for name, parameter in model.named_parameters():
        parameter.copy_(torch.tensor(INITIAL_PARAMETERS[name], dtype=torch.float64))
    logits_before = model(inputs, allowed).clone()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
loss_values = []
for _ in range(40):
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs, allowed)
    loss = nn.functional.cross_entropy(logits.reshape(-1, 4), targets.reshape(-1))
    loss_values.append(float(loss.detach()))
    loss.backward()
    optimizer.step()
losses = np.asarray(loss_values, dtype=np.float64)
with torch.no_grad():
    logits_after = model(inputs, allowed).clone()
probe_logits = torch.stack((logits_before[0, 0], logits_after[0, -1]))
predictions = logits_after.argmax(dim=-1)
EXPECTED_LOSS_ENDPOINTS = np.array([1.3967415632653304, 0.2672545835920813])
EXPECTED_PROBE_LOGITS = torch.tensor(
    [[0.03515, -0.029775, 0.035125, -0.0262875],
     [7.085651210268667, 3.4933718572789973, -9.903947554341626, -5.161603661279661]],
    dtype=torch.float64,
)

### Answer check

In [ ]:
assert inputs.shape == (1, 5, 4) and targets.shape == (1, 5)
assert logits_before.shape == logits_after.shape == (1, 5, 4)
assert losses.shape == (40,) and probe_logits.shape == (2, 4)
assert predictions.shape == (1, 5)
np.testing.assert_allclose(losses[[0, -1]], EXPECTED_LOSS_ENDPOINTS, atol=ATOL, rtol=RTOL)
assert torch.allclose(probe_logits, EXPECTED_PROBE_LOGITS, atol=ATOL, rtol=RTOL)
assert torch.equal(predictions, targets)
assert losses[-1] < losses[0]